# Playground — ลองเล่นฟังก์ชันที่เขียนไว้ใน Issue #2

ไฟล์นี้ไม่ใช่ EDA notebook ตัวจริง (อันนั้นจะทำใน Step 11) — ไฟล์นี้ไว้แค่ลองเรียกฟังก์ชัน
ที่เขียนไปแล้ว (Step 3-7) ทีละอัน ดูว่ามันทำงานยังไง คลิก ▶️ (Run) ที่แต่ละ cell
ไล่จากบนลงล่างได้เลย

## 1. config.py — ค่าคงที่ของโปรเจกต์

In [1]:
from goldforecast.config import TICKERS, HORIZONS

print("TICKERS:", TICKERS)
print("HORIZONS:", HORIZONS)

TICKERS: {'gold': 'GC=F', 'dxy': 'DX-Y.NYB', 'crude_oil': 'CL=F', 'vix': '^VIX', 'treasury_10y': '^TNX', 'sp500': '^GSPC'}
HORIZONS: [1, 5, 10]


## 2. features/indicators.py — ลองคำนวณ indicator บนข้อมูลราคาปลอมๆ

In [2]:
import pandas as pd
from goldforecast.features.indicators import moving_average, ema, rsi, macd

prices = pd.Series([100, 102, 101, 105, 108, 107, 110, 112, 111, 115], dtype=float)
prices

0    100.0
1    102.0
2    101.0
3    105.0
4    108.0
5    107.0
6    110.0
7    112.0
8    111.0
9    115.0
dtype: float64

In [3]:
moving_average(prices, window=3)

0           NaN
1           NaN
2    101.000000
3    102.666667
4    104.666667
5    106.666667
6    108.333333
7    109.666667
8    111.000000
9    112.666667
dtype: float64

In [4]:
ema(prices, span=3)

0    100.000000
1    101.000000
2    101.000000
3    103.000000
4    105.500000
5    106.250000
6    108.125000
7    110.062500
8    110.531250
9    112.765625
dtype: float64

In [5]:
rsi(prices, window=3)

0          NaN
1          NaN
2          NaN
3    85.714286
4    87.500000
5    87.500000
6    85.714286
7    83.333333
8    83.333333
9    85.714286
dtype: float64

In [6]:
macd_line, signal_line = macd(prices, fast=3, slow=6, signal=2)
pd.DataFrame({"price": prices, "macd_line": macd_line, "signal_line": signal_line})

,price,macd_line,signal_line
0,100.0,0.000000,0.000000
1,102.0,0.428571,0.285714
2,101.0,0.306122,0.299320
3,105.0,1.075802,0.816974
4,108.0,1.839858,1.498897
5,107.0,1.635613,1.590041
6,110.0,1.971867,1.844591
7,112.0,2.238833,2.107419
8,111.0,1.800059,1.902513
9,115.0,2.243346,2.129735


## 3. data/validate.py — ลองส่งข้อมูลดีกับข้อมูลเสียเข้าไปเช็ค

In [7]:
from goldforecast.data.validate import schema_check, missing_value_check, range_check, DataValidationError

good_df = pd.DataFrame({
    "Date": pd.date_range("2024-01-01", periods=3),
    "Open": [100.0, 101.0, 102.0],
    "High": [101.0, 102.0, 103.0],
    "Low": [99.0, 100.0, 101.0],
    "Close": [100.5, 101.5, 102.5],
    "Volume": [1000, 1100, 1200],
})

schema_check(good_df)
missing_value_check(good_df)
range_check(good_df, "Close", min_value=0)
print("ข้อมูลดี ผ่านหมดทุกเช็ค ไม่มีอะไรพิมพ์ออกมา (เพราะไม่ raise)")

ข้อมูลดี ผ่านหมดทุกเช็ค ไม่มีอะไรพิมพ์ออกมา (เพราะไม่ raise)


In [8]:
bad_df = good_df.copy()
bad_df.loc[0, "Low"] = -5.0  # ราคาติดลบ ไม่สมเหตุสมผล

try:
    range_check(bad_df, "Low", min_value=0)
except DataValidationError as e:
    print("โดน raise ตามคาด:", e)

โดน raise ตามคาด: Low has values below 0


## 4. data/merge.py — ลองรวมข้อมูล 2 source เข้าด้วยกัน

In [9]:
from goldforecast.data.merge import merge_sources

gold = pd.DataFrame({
    "Date": pd.date_range("2024-01-01", periods=3),
    "Open": [100.0, 101.0, 102.0], "High": [101.0, 102.0, 103.0],
    "Low": [99.0, 100.0, 101.0], "Close": [100.5, 101.5, 102.5],
    "Volume": [1000, 1100, 1200],
})
dxy = pd.DataFrame({
    "Date": pd.date_range("2024-01-01", periods=3),
    "Open": [90.0, 90.0, 91.0], "High": [90.0, 91.0, 91.0],
    "Low": [89.0, 90.0, 90.0], "Close": [90.0, 90.5, 91.0],
    "Volume": [500, 500, 500],
})

merge_sources({"gold": gold, "dxy": dxy})

,Date,Open,High,Low,Close,Volume,dxy_close
0,2024-01-01,100.0,101.0,99.0,100.5,1000,90.0
1,2024-01-02,101.0,102.0,100.0,101.5,1100,90.5
2,2024-01-03,102.0,103.0,101.0,102.5,1200,91.0


## 5. data/fetch.py — ดึงข้อมูลจริงจาก Yahoo Finance (ต้องมีเน็ต)

อันนี้ต่างจากข้างบน เพราะยิง network จริงไปหา Yahoo Finance ใช้เวลาสักครู่

In [10]:
from datetime import date, timedelta
from goldforecast.data.fetch import fetch_source

end = date.today().isoformat()
start = (date.today() - timedelta(days=14)).isoformat()

fetch_source("GC=F", start=start, end=end)

Price,Date,Open,High,Low,Close,Volume
0,2026-08-31,4430.000000,4466.899902,4410.899902,4431.100098,360
1,2026-09-01,4402.000000,4402.000000,4329.100098,4348.000000,191
2,2026-09-02,4328.000000,4390.200195,4292.200195,4366.299805,72
3,2026-09-03,4426.299805,4510.000000,4426.000000,4491.700195,16
4,2026-09-04,4429.799805,4429.799805,4429.799805,4429.799805,145
5,2026-09-08,4391.899902,4406.100098,4384.399902,4393.899902,215
6,2026-09-09,4398.700195,4416.000000,4397.399902,4416.000000,86
7,2026-09-10,4418.899902,4420.000000,4330.700195,4364.500000,73
8,2026-09-11,4365.799805,4389.500000,4365.799805,4366.200195,73
